In [4]:
import pickle
import numpy as np
import pandas as pd

import sys

sys.path.append("..")
from cd_zoo.tools.scoring_tools import score
from robustness_experiments.plot import renames

# Calculate performance: 

In [ ]:
# Load the labels for the rivers dataset. 
# Check the cd_zoo causal rivers integration for more details on how to generate the dataset.
_, Y = pickle.load(
    open(
        "path_to/release_random_5_tcd_arena.p",
        "rb",
    )
)
Y = np.stack(Y[:500])

In [ ]:
# individual results (prepared with the scoring functionalities in TCD-Arena)
individual = pd.read_csv(
    "release_ensemble_rivers/extracted/SG_max/rivers.csv",
    index_col=0,
)
individual.index = individual.index.str[-27:]

In [17]:
individual["SHD individual"]

ctions/rivers/cp//best_run/    0.8508
ons/rivers/pcmci//best_run/    0.9777
ivers/ntsnotears//best_run/    0.7583
rivers/pcmciplus//best_run/    0.8353
rivers/varlingam//best_run/    0.7745
rivers/dynotears//best_run/    0.7146
ns/rivers/fpcmci//best_run/    0.8397
/rivers/svarrfci//best_run/    0.8570
tions/rivers/var//best_run/    0.7606
direct_crosscorr//best_run/    0.7984
Name: SHD individual, dtype: float64

# Performance of meaning: 

In [19]:
# Loading the processed predictions from generate_rivers_dataset
preds = pickle.load(
    open(
        "processed_predictions.p",
        "rb",
    )
)

In [20]:
%%capture
# Calculate the mean performance by meaning over method axis
mean = score(
    Y,
    preds.mean(axis=1).max(axis=-1),
    instant_labs=None,
    instant_preds=None,
    per_sample_metrics=True,
    remove_autoregressive_for_lagged=True,
).loc["SHD individual"]

In [22]:
mean

SG_max    0.6592
Name: SHD individual, dtype: float64

# Ensemble performances:

In [ ]:
# Note, we only need the small ensembles as our samples only have 5 variables.
linear = "release_ensemble_rivers/release_rivers_ensemble_performance/SimpleLinear/wcg_wcg/small/predictions_2026-02-28_07-46-09-775607.pkl"
mlp = "release_ensemble_rivers/release_rivers_ensemble_performance/SimpleMLP/wcg_wcg/small/predictions_2026-03-01_17-59-15-747814.pkl"
transformer = "release_ensemble_rivers/release_rivers_ensemble_performance/SimpleTransformer/wcg_wcg/small/predictions_2026-02-28_06-53-05-310897.pkl"


linear = pickle.load(open(linear, "rb"))
mlp = pickle.load(open(mlp, "rb"))
transformer = pickle.load(open(transformer, "rb"))

# maximum of last dimension:
linear = linear.max(axis=-1)
mlp = mlp.max(axis=-1)
transformer = transformer.max(axis=-1)


In [18]:
%%capture
linear_res = score(
    Y,
    linear,
    instant_labs=None,
    instant_preds=None,
    per_sample_metrics=True,
    remove_autoregressive_for_lagged=True,
).loc["SHD individual"]

In [19]:
%%capture
mlp_res = score(
    Y,
    mlp,
    instant_labs=None,
    instant_preds=None,
    per_sample_metrics=True,
    remove_autoregressive_for_lagged=True,
).loc["SHD individual"]

In [20]:
%%capture
transformer_res = score(
    Y,
    transformer,
    instant_labs=None,
    instant_preds=None,
    per_sample_metrics=True,
    remove_autoregressive_for_lagged=True,
).loc["SHD individual"]

In [21]:
res_table = individual["SHD individual"]

res_table.index = [x.split("/")[-4] for x in res_table.index]
res_table.index = [renames[x] for x in res_table.index]

In [22]:
res_table["Ensemble (Mean)"] = mean.values[0]
res_table["Ensemble (Linear)"] = linear_res.values[0]
res_table["Ensemble (MLP)"] = mlp_res.values[0]
res_table["Ensemble (Transformer)"] = transformer_res.values[0]

/tmp/ipykernel_101381/1117565627.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res_table["Ensemble (Mean)"] = mean.values[0]


In [23]:
print(pd.DataFrame(res_table).round(3).to_latex())

\begin{tabular}{lr}
\toprule
{} &  SHD individual \\
\midrule
CausalPretraining      &           0.851 \\
PCMCI                  &           0.978 \\
Nts-Notears            &           0.758 \\
PCMCI+                 &           0.835 \\
Varlingam              &           0.774 \\
Dynotears              &           0.715 \\
F-PCMCI                &           0.840 \\
SVAR-RFCI              &           0.857 \\
GVAR                   &           0.761 \\
CrossCorrelation       &           0.798 \\
Ensemble (Mean)        &           0.659 \\
Ensemble (Linear)      &           0.666 \\
Ensemble (MLP)         &           0.685 \\
Ensemble (Transformer) &           0.686 \\
\bottomrule
\end{tabular}



/tmp/ipykernel_101381/3529877854.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(pd.DataFrame(res_table).round(3).to_latex())
